# PolicyRec v1.1 — 공통 컬럼 정규화 파이프라인

## 이 노트북의 역할

v1.0이 "원본을 잃지 않는 것"에 집중했다면,  
v1.1은 **추천·검색·필터에 실제로 쓸 수 있는 형태로 데이터를 정리하는 기준을 잡는 것**이 목적입니다.

즉, 이 노트북은 다음 두 가지를 동시에 합니다.

1. **기준 문서** — 컬럼 매핑, 날짜 형식, 결측값 처리 기준을 코드로 명확히 정의합니다.
2. **실행 코드** — 실제 데이터(`combined_raw_columns.csv`)가 생기면 바로 돌릴 수 있는 파이프라인을 미리 작성합니다.

## 현재 상태와 한계

> **현재 이 노트북은 실제 API 데이터 없이 작성되었습니다.**

v1.0의 `fetch.py` + `app` 패키지가 완성되어 `data/raw`에 JSON 파일이 생기기 전까지는,  
아래 mock 데이터를 사용해 함수 동작과 변환 기준을 검증합니다.

**실제 데이터로 전환하려면:**
- `USE_MOCK = False`로 변경하면 됩니다. (0번 셀 설정값 참고)
- 나머지 코드는 수정 없이 그대로 사용 가능합니다.

## 버전별 역할 요약

| 버전 | 목표 | 결과물 | 담당 |
| :--- | :--- | :--- | :--- |
| `v1.0` | API 원본 보존 및 출처 추적 | `combined_raw_columns.csv` | 완료 |
| **`v1.1`** | **공통 컬럼 정규화 기준 정의 + 파이프라인** | **`combined_normalized_v1_1.csv`** | **이 노트북** |
| `v2` | 첨부파일(PDF/HWP) 본문 추가 | 목록 + 첨부파일 연결 데이터 | 예정 |

In [ ]:
# ============================================================
# 0. 기본 설정
# ============================================================
# 자주 바꿀 값은 이 셀에 모아 두었습니다.

from pathlib import Path
import re
import json
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(v): print(v)


# ============================================================
# 사용자가 자주 바꿀 설정값
# ============================================================

# True  : mock 데이터로 실행 (실제 CSV 없어도 동작)
# False : 실제 combined_raw_columns.csv 사용
USE_MOCK = True

SOURCES = ["biz", "kst", "youth"]

SOURCE_LABELS = {
    "biz":   "Bizinfo",
    "kst":   "K-Startup",
    "youth": "Youthcenter",
}

CSV_ENCODING    = "utf-8-sig"
PREVIEW_ROW_COUNT = 3

# ============================================================
# 결측값 처리 기준
# ============================================================
# 아래 값들은 팀 논의를 통해 변경할 수 있습니다.

DEFAULT_REGION   = "전국"   # region 없을 때
DEFAULT_AGE_MIN  = 0        # 연령 하한 없을 때
DEFAULT_AGE_MAX  = 99       # 연령 상한 없을 때
DEFAULT_CATEGORY = "기타"   # category 없을 때

# ============================================================
# 경로 설정
# ============================================================

PROJECT_ROOT    = Path.cwd()
CLEAN_ROOT      = PROJECT_ROOT / "data" / "clean"
RAW_MERGED_FILE = CLEAN_ROOT / "combined_raw_columns.csv"       # v1.0 결과 (입력)
NORMALIZED_FILE = CLEAN_ROOT / "combined_normalized_v1_1.csv"   # v1.1 결과 (출력)

print("설정 완료")
print(f"  USE_MOCK     : {USE_MOCK}")
print(f"  입력 파일    : {RAW_MERGED_FILE}")
print(f"  출력 파일    : {NORMALIZED_FILE}")

## 공통 스키마 정의

아래 두 표는 이 노트북에서 가장 중요한 기준입니다.

### 최종 공통 컬럼 (index.html 기준)

| 컬럼명 | 설명 | 비고 |
| :--- | :--- | :--- |
| `source` | 출처 코드 | biz / kst / youth |
| `source_id` | 원본 고유 ID | |
| `title` | 공고/정책 제목 | |
| `summary` | 요약 | |
| `category` | 분야 | 없으면 `기타` |
| `benefit_type` | 지원 형태 | biz/kst/youth 모두 없음 → 추후 확인 필요 |
| `region` | 지역 | 없으면 `전국` |
| `target_group` | 대상 | |
| `target_age_min` | 최소 연령 | 없으면 `0` |
| `target_age_max` | 최대 연령 | 없으면 `99` |
| `start_date` | 신청 시작일 | **YYYY-MM-DD 고정** |
| `end_date` | 신청 마감일 | **YYYY-MM-DD 고정** |
| `detail_url` | 상세 URL | |

### source별 원본 컬럼 → 공통 컬럼 매핑

| 공통 컬럼 | Bizinfo | K-Startup | Youthcenter |
| :--- | :--- | :--- | :--- |
| `source_id` | `pblancId` | `pbanc_sn` | `plcyNo` |
| `title` | `pblancNm` | `biz_pbanc_nm` | `plcyNm` |
| `summary` | `bsnsSumryCn` | `pbanc_ctnt` | `plcyExplnCn` |
| `category` | `pldirSportRealmLclasCodeNm` | `supt_biz_clsfc` | `lclsfNm` |
| `benefit_type` | ❌ 없음 | ❌ 없음 | ❌ 없음 |
| `region` | `jrsdInsttNm` | `supt_regin` | `zipCd` |
| `target_group` | `trgetNm` | `aply_trgt` | `ptcpPrpTrgtCn` |
| `target_age` | ❌ 없음 → 0~99 | `biz_trgt_age` (파싱 필요) | min/max 컬럼 분리 |
| `start_date` | `reqstBeginEndDe` (분리 필요) | `pbanc_rcpt_bgng_dt` | `bizPrdBgngYmd` |
| `end_date` | `reqstBeginEndDe` (분리 필요) | `pbanc_rcpt_end_dt` | `bizPrdEndYmd` |
| `detail_url` | `pblancUrl` | `detl_pg_url` | `aplyUrlAddr` |

> ⚠️ **`benefit_type`은 세 source 모두 직접 대응되는 컬럼이 없습니다.**  
> 실제 데이터 확인 후 파싱 방법을 별도로 정해야 합니다.

In [ ]:
# ============================================================
# 공통 스키마 및 매핑 테이블
# ============================================================

COMMON_COLUMNS = [
    "source", "source_id", "title", "summary", "category",
    "benefit_type", "region", "target_group",
    "target_age_min", "target_age_max",
    "start_date", "end_date", "detail_url",
]

# 원본 컬럼명 → 공통 컬럼명 매핑
# None : 해당 source에 대응 컬럼 없음 → 기본값 처리
COLUMN_MAP = {
    "biz": {
        "source_id":    "pblancId",
        "title":        "pblancNm",
        "summary":      "bsnsSumryCn",
        "category":     "pldirSportRealmLclasCodeNm",
        "benefit_type": None,
        "region":       "jrsdInsttNm",
        "target_group": "trgetNm",
        "target_age":   None,                   # 없음 → 0~99
        "start_date":   "reqstBeginEndDe",       # "YYYY-MM-DD ~ YYYY-MM-DD" 분리 필요
        "end_date":     "reqstBeginEndDe",
        "detail_url":   "pblancUrl",
    },
    "kst": {
        "source_id":    "pbanc_sn",
        "title":        "biz_pbanc_nm",
        "summary":      "pbanc_ctnt",
        "category":     "supt_biz_clsfc",
        "benefit_type": None,
        "region":       "supt_regin",
        "target_group": "aply_trgt",
        "target_age":   "biz_trgt_age",          # "만 19~39세" 형태 → 파싱 필요
        "start_date":   "pbanc_rcpt_bgng_dt",
        "end_date":     "pbanc_rcpt_end_dt",
        "detail_url":   "detl_pg_url",
    },
    "youth": {
        "source_id":    "plcyNo",
        "title":        "plcyNm",
        "summary":      "plcyExplnCn",
        "category":     "lclsfNm",
        "benefit_type": None,
        "region":       "zipCd",
        "target_group": "ptcpPrpTrgtCn",
        "target_age":   "_age_range",            # min/max 컬럼 자동 탐색
        "start_date":   "bizPrdBgngYmd",
        "end_date":     "bizPrdEndYmd",
        "detail_url":   "aplyUrlAddr",
    },
}

print("스키마 정의 완료")
print("공통 컬럼:", COMMON_COLUMNS)

## Mock 데이터 정의

실제 API 응답이 없는 현재 상태에서 함수 동작을 검증하기 위한 샘플 데이터입니다.

각 source의 **실제 API 응답 구조와 컬럼명**을 그대로 모방했습니다.  
실제 데이터로 전환할 때 이 셀은 무시됩니다. (`USE_MOCK = False`)

> ⚠️ **검증이 필요한 항목**  
> mock 데이터에서 의도적으로 다양한 케이스를 넣었습니다.
> - 날짜: `20260420` / `2026-04-20` / `2026-04-01 ~ 2026-05-31` 혼재
> - 연령: `만 19~39세` / `39세 이하` / `제한 없음` / None 혼재
> - 결측값: region, category 일부 None 포함

In [ ]:
# ============================================================
# Mock 데이터 (USE_MOCK = True일 때만 사용)
# ============================================================
# 실제 API 응답 컬럼명을 그대로 사용합니다.
# v1.0의 combined_raw_columns.csv와 동일한 구조입니다.

MOCK_ROWS = [
    # ── Bizinfo ──────────────────────────────────────────────
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "_source_file": "data/raw/biz/sample.json", "_source_row_number": 1,
        "pblancId": "BIZ001",
        "pblancNm": "2026년 소상공인 경영안정자금 지원",
        "bsnsSumryCn": "소상공인의 경영 안정을 위한 저금리 융자 지원 사업입니다.",
        "pldirSportRealmLclasCodeNm": "금융",
        "jrsdInsttNm": "중소벤처기업부",
        "trgetNm": "소상공인",
        "reqstBeginEndDe": "2026-04-01 ~ 2026-05-31",  # 분리 필요
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ001",
    },
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "_source_file": "data/raw/biz/sample.json", "_source_row_number": 2,
        "pblancId": "BIZ002",
        "pblancNm": "청년 창업 도약 패키지",
        "bsnsSumryCn": "창업 3년 이내 청년 기업 대상 사업화 자금 및 멘토링 지원.",
        "pldirSportRealmLclasCodeNm": None,             # category 없음 → 기타
        "jrsdInsttNm": None,                            # region 없음 → 전국
        "trgetNm": "창업 3년 이내 기업",
        "reqstBeginEndDe": "2026.05.01~2026.06.30",    # 점 구분자 케이스
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ002",
    },

    # ── K-Startup ─────────────────────────────────────────────
    {
        "_source": "kst", "_source_name": "K-Startup",
        "_source_file": "data/raw/kst/sample.json", "_source_row_number": 1,
        "pbanc_sn": "KST001",
        "biz_pbanc_nm": "초기창업패키지",
        "pbanc_ctnt": "창업 3년 이내 초기 창업기업 대상 사업화 자금 지원.",
        "supt_biz_clsfc": "창업",
        "supt_regin": "전국",
        "aply_trgt": "창업 3년 이내 기업",
        "biz_trgt_age": "만 19~39세",                  # 파싱 필요
        "pbanc_rcpt_bgng_dt": "2026-04-10",
        "pbanc_rcpt_end_dt": "2026-05-10",
        "detl_pg_url": "https://www.k-startup.go.kr/KST001",
    },
    {
        "_source": "kst", "_source_name": "K-Startup",
        "_source_file": "data/raw/kst/sample.json", "_source_row_number": 2,
        "pbanc_sn": "KST002",
        "biz_pbanc_nm": "예비창업패키지",
        "pbanc_ctnt": "혁신적인 기술 창업 아이디어 보유 예비창업자 지원.",
        "supt_biz_clsfc": "창업",
        "supt_regin": "서울",
        "aply_trgt": "예비창업자",
        "biz_trgt_age": "39세 이하",                   # 단방향 파싱 케이스
        "pbanc_rcpt_bgng_dt": "20260415",              # 8자리 숫자 케이스
        "pbanc_rcpt_end_dt": "20260515",
        "detl_pg_url": "https://www.k-startup.go.kr/KST002",
    },
    {
        "_source": "kst", "_source_name": "K-Startup",
        "_source_file": "data/raw/kst/sample.json", "_source_row_number": 3,
        "pbanc_sn": "KST003",
        "biz_pbanc_nm": "글로벌 액셀러레이팅 프로그램",
        "pbanc_ctnt": "해외 진출을 희망하는 스타트업 대상 글로벌 네트워킹 지원.",
        "supt_biz_clsfc": "글로벌",
        "supt_regin": None,
        "aply_trgt": "스타트업",
        "biz_trgt_age": "제한 없음",                   # 전연령 케이스
        "pbanc_rcpt_bgng_dt": "2026-05-01",
        "pbanc_rcpt_end_dt": "2026-06-01",
        "detl_pg_url": "https://www.k-startup.go.kr/KST003",
    },

    # ── Youthcenter ───────────────────────────────────────────
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "_source_file": "data/raw/youth/sample.json", "_source_row_number": 1,
        "plcyNo": "YTH001",
        "plcyNm": "청년 월세 한시 특별지원",
        "plcyExplnCn": "독립 거주 청년의 주거비 부담 완화를 위한 월세 지원.",
        "lclsfNm": "주거",
        "zipCd": "서울특별시",
        "ptcpPrpTrgtCn": "만 19~34세 독립거주 청년",
        "ageMin": "19",
        "ageMax": "34",
        "bizPrdBgngYmd": "20260401",                   # 8자리 숫자 케이스
        "bizPrdEndYmd": "20261231",
        "aplyUrlAddr": "https://www.youthcenter.go.kr/YTH001",
    },
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "_source_file": "data/raw/youth/sample.json", "_source_row_number": 2,
        "plcyNo": "YTH002",
        "plcyNm": "청년 내일저축계좌",
        "plcyExplnCn": "저소득 청년의 자산 형성을 돕는 매칭 저축 지원 사업.",
        "lclsfNm": "금융",
        "zipCd": None,                                  # region 없음 → 전국
        "ptcpPrpTrgtCn": "일하는 저소득 청년",
        "ageMin": None,                                 # 연령 없음 → 기본값
        "ageMax": "34",
        "bizPrdBgngYmd": "2026-05-02",
        "bizPrdEndYmd": "2026-05-31",
        "aplyUrlAddr": "https://www.youthcenter.go.kr/YTH002",
    },
]

print(f"mock 데이터 준비 완료: {len(MOCK_ROWS)}건")
print("source별:", {s: sum(1 for r in MOCK_ROWS if r['_source']==s) for s in SOURCES})

## Helper 함수 정의

| 함수명 | 역할 |
| :--- | :--- |
| `normalize_date()` | 다양한 날짜 형식 → `YYYY-MM-DD` |
| `parse_biz_date()` | Bizinfo `reqstBeginEndDe` (범위값) → start / end 분리 |
| `parse_kst_age()` | K-Startup `biz_trgt_age` (텍스트) → min / max 숫자 |
| `parse_youth_age()` | Youthcenter min/max 컬럼 → 숫자 추출 |
| `clean_bizinfo()` | Bizinfo DataFrame → 공통 스키마 |
| `clean_kst()` | K-Startup DataFrame → 공통 스키마 |
| `clean_youth()` | Youthcenter DataFrame → 공통 스키마 |

In [ ]:
# ============================================================
# 날짜 변환 함수
# ============================================================

def normalize_date(value):
    """
    다양한 날짜 형식을 YYYY-MM-DD 문자열로 변환합니다.

    처리 가능한 형식:
      "20260420"       → "2026-04-20"   (8자리 숫자)
      "2026-04-20"     → "2026-04-20"   (이미 올바른 형식)
      "2026.04.20"     → "2026-04-20"   (점 구분자)
      "2026/04/20"     → "2026-04-20"   (슬래시 구분자)
      None / 빈값      → None

    ※ Bizinfo 범위값("YYYY-MM-DD ~ YYYY-MM-DD")은
      parse_biz_date()에서 먼저 분리한 뒤 이 함수로 전달합니다.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None

    s = str(value).strip()

    # 8자리 숫자: YYYYMMDD
    if re.fullmatch(r"\d{8}", s):
        return f"{s[:4]}-{s[4:6]}-{s[6:8]}"

    # 점·슬래시 → 하이픈
    s = re.sub(r"[./]", "-", s)

    # YYYY-MM-DD 형식 최종 확인
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return s

    # 알 수 없는 형식
    return None


def parse_biz_date(value):
    """
    Bizinfo reqstBeginEndDe를 (start_date, end_date) 튜플로 분리합니다.

    입력 예시:
      "2026-04-01 ~ 2026-05-31"  → ("2026-04-01", "2026-05-31")
      "2026.05.01~2026.06.30"    → ("2026-05-01", "2026-06-30")
      None                       → (None, None)
    """
    if pd.isna(value) or str(value).strip() == "":
        return None, None

    parts = re.split(r"\s*~\s*", str(value).strip())
    start = normalize_date(parts[0]) if len(parts) >= 1 else None
    end   = normalize_date(parts[1]) if len(parts) >= 2 else None
    return start, end


# ============================================================
# 연령 파싱 함수
# ============================================================

def parse_kst_age(value):
    """
    K-Startup biz_trgt_age 텍스트에서 (min, max) 숫자 튜플을 추출합니다.

    입력 예시:
      "만 19~39세"  → (19, 39)
      "39세 이하"   → (0,  39)
      "19세 이상"   → (19, 99)
      "제한 없음"   → (0,  99)
      None          → (0,  99)
    """
    if pd.isna(value) or str(value).strip() == "":
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    s = str(value).strip()

    m = re.search(r"(\d+)\s*~\s*(\d+)", s)
    if m:
        return int(m.group(1)), int(m.group(2))

    m = re.search(r"(\d+)\s*세?\s*이하", s)
    if m:
        return DEFAULT_AGE_MIN, int(m.group(1))

    m = re.search(r"(\d+)\s*세?\s*이상", s)
    if m:
        return int(m.group(1)), DEFAULT_AGE_MAX

    return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX


def parse_youth_age(min_val, max_val):
    """
    Youthcenter ageMin / ageMax 컬럼에서 숫자를 추출합니다.

    입력 예시:
      ("19", "34")    → (19, 34)
      (None, "34")    → (0,  34)
      (None, None)    → (0,  99)
    """
    try:
        mn = int(float(min_val)) if not pd.isna(min_val) else DEFAULT_AGE_MIN
    except (ValueError, TypeError):
        mn = DEFAULT_AGE_MIN

    try:
        mx = int(float(max_val)) if not pd.isna(max_val) else DEFAULT_AGE_MAX
    except (ValueError, TypeError):
        mx = DEFAULT_AGE_MAX

    return mn, mx


print("날짜·연령 함수 정의 완료")

In [ ]:
# ============================================================
# source별 정규화 함수
# ============================================================
# source마다 함수를 분리해두면:
# - 한 source의 원본 컬럼이 바뀌어도 나머지에 영향이 없습니다.
# - 나중에 source가 추가되면 함수만 하나 더 만들면 됩니다.

def clean_bizinfo(df):
    """
    Bizinfo DataFrame을 공통 스키마로 변환합니다.

    주요 처리:
    - reqstBeginEndDe : "YYYY-MM-DD ~ YYYY-MM-DD" → start_date / end_date 분리
    - target_age      : 컬럼 없음 → 0 / 99 기본값
    - benefit_type    : 컬럼 없음 → None (실제 데이터 확인 후 보완 필요)
    """
    m   = COLUMN_MAP["biz"]
    out = pd.DataFrame()

    out["source"]       = df["_source"]
    out["source_id"]    = df.get(m["source_id"])
    out["title"]        = df.get(m["title"])
    out["summary"]      = df.get(m["summary"])
    out["category"]     = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)
    out["benefit_type"] = None  # ← 실제 데이터 확인 후 보완
    out["region"]       = df.get(m["region"], pd.Series([DEFAULT_REGION]*len(df))).fillna(DEFAULT_REGION)
    out["target_group"] = df.get(m["target_group"])
    out["target_age_min"] = DEFAULT_AGE_MIN
    out["target_age_max"] = DEFAULT_AGE_MAX

    date_col = m["start_date"]
    if date_col in df.columns:
        parsed = df[date_col].apply(parse_biz_date)
        out["start_date"] = parsed.apply(lambda t: t[0])
        out["end_date"]   = parsed.apply(lambda t: t[1])
    else:
        out["start_date"] = None
        out["end_date"]   = None

    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


def clean_kst(df):
    """
    K-Startup DataFrame을 공통 스키마로 변환합니다.

    주요 처리:
    - biz_trgt_age    : "만 19~39세" 등 텍스트 → target_age_min / max 파싱
    - 날짜 컬럼       : 8자리 숫자 / YYYY-MM-DD 혼재 → normalize_date()
    - benefit_type    : 컬럼 없음 → None
    """
    m   = COLUMN_MAP["kst"]
    out = pd.DataFrame()

    out["source"]       = df["_source"]
    out["source_id"]    = df.get(m["source_id"])
    out["title"]        = df.get(m["title"])
    out["summary"]      = df.get(m["summary"])
    out["category"]     = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)
    out["benefit_type"] = None
    out["region"]       = df.get(m["region"], pd.Series([DEFAULT_REGION]*len(df))).fillna(DEFAULT_REGION)
    out["target_group"] = df.get(m["target_group"])

    age_col = m["target_age"]
    if age_col in df.columns:
        ages = df[age_col].apply(parse_kst_age)
        out["target_age_min"] = ages.apply(lambda t: t[0])
        out["target_age_max"] = ages.apply(lambda t: t[1])
    else:
        out["target_age_min"] = DEFAULT_AGE_MIN
        out["target_age_max"] = DEFAULT_AGE_MAX

    out["start_date"] = df.get(m["start_date"], pd.Series([None]*len(df))).apply(normalize_date)
    out["end_date"]   = df.get(m["end_date"],   pd.Series([None]*len(df))).apply(normalize_date)
    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


def clean_youth(df):
    """
    Youthcenter DataFrame을 공통 스키마로 변환합니다.

    주요 처리:
    - 연령 컬럼       : ageMin / ageMax 컬럼명 자동 탐색 후 숫자 추출
    - 날짜 컬럼       : 8자리 숫자 / YYYY-MM-DD 혼재 → normalize_date()
    - benefit_type    : 컬럼 없음 → None

    ※ 연령 컬럼명이 실제 API 응답에서 다를 수 있습니다.
       아래 age_min_candidates / age_max_candidates 로그를 확인하세요.
    """
    m   = COLUMN_MAP["youth"]
    out = pd.DataFrame()

    out["source"]       = df["_source"]
    out["source_id"]    = df.get(m["source_id"])
    out["title"]        = df.get(m["title"])
    out["summary"]      = df.get(m["summary"])
    out["category"]     = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)
    out["benefit_type"] = None
    out["region"]       = df.get(m["region"], pd.Series([DEFAULT_REGION]*len(df))).fillna(DEFAULT_REGION)
    out["target_group"] = df.get(m["target_group"])

    # 연령 컬럼 자동 탐색 (컬럼명이 바뀌어도 대응 가능하도록)
    min_cands = [c for c in df.columns if "min" in c.lower() and "age" in c.lower()]
    max_cands = [c for c in df.columns if "max" in c.lower() and "age" in c.lower()]
    print(f"  [youth 연령 컬럼] min 후보: {min_cands}, max 후보: {max_cands}")

    min_col = min_cands[0] if min_cands else None
    max_col = max_cands[0] if max_cands else None

    if min_col or max_col:
        min_s = df[min_col] if min_col else pd.Series([None]*len(df))
        max_s = df[max_col] if max_col else pd.Series([None]*len(df))
        ages  = pd.concat([min_s.reset_index(drop=True),
                           max_s.reset_index(drop=True)], axis=1)
        ages.columns = ["_mn", "_mx"]
        out["target_age_min"] = ages.apply(lambda r: parse_youth_age(r["_mn"], r["_mx"])[0], axis=1)
        out["target_age_max"] = ages.apply(lambda r: parse_youth_age(r["_mn"], r["_mx"])[1], axis=1)
    else:
        print("  [youth 연령 컬럼] 탐색 실패 → 기본값(0~99) 적용")
        out["target_age_min"] = DEFAULT_AGE_MIN
        out["target_age_max"] = DEFAULT_AGE_MAX

    out["start_date"] = df.get(m["start_date"], pd.Series([None]*len(df))).apply(normalize_date)
    out["end_date"]   = df.get(m["end_date"],   pd.Series([None]*len(df))).apply(normalize_date)
    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


print("source별 정규화 함수 정의 완료")
print("정의된 함수: clean_bizinfo(), clean_kst(), clean_youth()")

## 1단계. 데이터 불러오기

- `USE_MOCK = True`  → mock 데이터 사용 (실제 CSV 없어도 동작)
- `USE_MOCK = False` → `combined_raw_columns.csv` 사용 (v1.0 실행 후 가능)

In [ ]:
if USE_MOCK:
    print("[USE_MOCK=True] mock 데이터를 사용합니다.")
    raw_df = pd.DataFrame(MOCK_ROWS, dtype=str).replace("None", pd.NA)
    # mock에서 None으로 넣은 값이 문자열 "None"이 되지 않도록 처리
    for col in raw_df.columns:
        raw_df[col] = raw_df[col].apply(lambda x: None if x == "None" or x is None else x)
else:
    print("[USE_MOCK=False] 실제 CSV를 사용합니다.")
    if not RAW_MERGED_FILE.exists():
        raise FileNotFoundError(
            f"파일 없음: {RAW_MERGED_FILE}\n"
            "PolicyRec_v1.0.ipynb를 먼저 실행해 주세요."
        )
    raw_df = pd.read_csv(RAW_MERGED_FILE, encoding=CSV_ENCODING, dtype=str)

print(f"불러온 데이터: {raw_df.shape[0]}행 × {raw_df.shape[1]}열")
print("\nsource별 행 개수:")
display(raw_df["_source"].value_counts().rename_axis("source").reset_index(name="row_count"))

## 2단계. 매핑 컬럼 존재 여부 사전 확인

정규화 실행 전, 매핑 대상 컬럼이 실제로 있는지 확인합니다.  
`존재: False`인 항목은 해당 source에 컬럼이 없거나 컬럼명이 다른 경우입니다.  
→ `COLUMN_MAP`에서 컬럼명을 수정하거나, 기본값 처리 방식을 조정하세요.

In [ ]:
source_dfs = {
    s: raw_df[raw_df["_source"] == s].copy().reset_index(drop=True)
    for s in SOURCES
}

check_rows = []
for source_name in SOURCES:
    df = source_dfs[source_name]
    for common_col, raw_col in COLUMN_MAP[source_name].items():
        if raw_col is None:
            check_rows.append({"source": source_name, "공통 컬럼": common_col,
                                "원본 컬럼": "(없음)", "존재": "—", "non-null 수": "—"})
        elif raw_col in df.columns:
            nn = df[raw_col].notna().sum()
            check_rows.append({"source": source_name, "공통 컬럼": common_col,
                                "원본 컬럼": raw_col, "존재": "✓", "non-null 수": nn})
        else:
            check_rows.append({"source": source_name, "공통 컬럼": common_col,
                                "원본 컬럼": raw_col, "존재": "❌ 없음", "non-null 수": 0})

display(pd.DataFrame(check_rows))

## 3단계. 변환 함수 단위 검증

정규화 실행 전에 각 함수가 의도대로 동작하는지 샘플로 확인합니다.

In [ ]:
# ---- 날짜 변환 검증 ----
print("=== normalize_date() 검증 ===")
date_cases = [
    "20260420",
    "2026-04-20",
    "2026.04.20",
    "2026/04/20",
    None,
    "",
]
for v in date_cases:
    print(f"  {str(v):20s} → {normalize_date(v)}")

print("\n=== parse_biz_date() 검증 ===")
biz_cases = [
    "2026-04-01 ~ 2026-05-31",
    "2026.05.01~2026.06.30",
    None,
]
for v in biz_cases:
    print(f"  {str(v):35s} → {parse_biz_date(v)}")

# ---- 연령 파싱 검증 ----
print("\n=== parse_kst_age() 검증 ===")
age_cases = ["만 19~39세", "39세 이하", "19세 이상", "제한 없음", None]
for v in age_cases:
    print(f"  {str(v):15s} → {parse_kst_age(v)}")

print("\n=== parse_youth_age() 검증 ===")
youth_age_cases = [("19", "34"), (None, "34"), (None, None), ("abc", "34")]
for mn, mx in youth_age_cases:
    print(f"  ({mn}, {mx}):  → {parse_youth_age(mn, mx)}")

## 4단계. source별 정규화 실행

In [ ]:
clean_funcs = {
    "biz":   clean_bizinfo,
    "kst":   clean_kst,
    "youth": clean_youth,
}

normalized_frames = []

for source_name in SOURCES:
    df = source_dfs[source_name]
    if df.empty:
        print(f"[{source_name}] 데이터 없음. 건너뜁니다.")
        continue

    norm_df = clean_funcs[source_name](df)
    normalized_frames.append(norm_df)

    print(f"\n===== {SOURCE_LABELS[source_name]} 정규화 결과 ({PREVIEW_ROW_COUNT}행) =====")
    display(norm_df.head(PREVIEW_ROW_COUNT))

## 5단계. 통합 및 CSV 저장

In [ ]:
normalized_df = pd.concat(normalized_frames, ignore_index=True)

CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
normalized_df.to_csv(NORMALIZED_FILE, index=False, encoding=CSV_ENCODING)

print("저장 완료:", NORMALIZED_FILE)
print("전체 행/열:", normalized_df.shape)
print("\nsource별 행 개수:")
display(normalized_df["source"].value_counts().rename_axis("source").reset_index(name="row_count"))
print("\n전체 미리보기:")
display(normalized_df)

## 6단계. 결과 검증

In [ ]:
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

# ---- 결측 비율 ----
print("=== 컬럼별 결측 비율 ===")
null_df = normalized_df.isna().mean().mul(100).round(1).rename("null%").reset_index()
null_df.columns = ["column", "null%"]
display(null_df)

# ---- 날짜 형식 이상값 ----
print("\n=== 날짜 형식 이상값 (YYYY-MM-DD 미준수) ===")
for date_col in ["start_date", "end_date"]:
    bad = normalized_df[date_col].notna() & \
          ~normalized_df[date_col].apply(lambda v: bool(DATE_PATTERN.match(str(v))))
    if bad.sum():
        print(f"[{date_col}] 이상값 {bad.sum()}건:")
        display(normalized_df.loc[bad, ["source", "source_id", date_col]])
    else:
        print(f"[{date_col}] 이상값 없음 ✓")

# ---- 연령 이상값 ----
print("\n=== 연령 이상값 (min > max) ===")
age_bad = (
    normalized_df["target_age_min"].notna() &
    normalized_df["target_age_max"].notna() &
    (normalized_df["target_age_min"].astype(float) > normalized_df["target_age_max"].astype(float))
)
if age_bad.sum():
    display(normalized_df.loc[age_bad, ["source","source_id","target_age_min","target_age_max"]])
else:
    print("이상값 없음 ✓")

## 정리와 다음 작업

### v1.1에서 완료된 것
1. source별 정규화 함수 분리 (`clean_bizinfo`, `clean_kst`, `clean_youth`)
2. 날짜 형식 `YYYY-MM-DD` 통일 (8자리 숫자, 점·슬래시 구분자, Bizinfo 범위값 처리 포함)
3. 연령 파싱 및 `target_age_min` / `target_age_max` 분리
4. 결측값 기본값 처리 (region → `전국`, age → `0~99`, category → `기타`)
5. `combined_normalized_v1_1.csv` 저장 + 이상값 자동 탐지

### 실제 데이터로 전환하려면
1. `fetch.py` + `app` 패키지 완성 → `data/raw` JSON 파일 생성
2. v1.0 노트북 실행 → `combined_raw_columns.csv` 생성
3. 이 노트북 0번 셀에서 `USE_MOCK = False`로 변경 후 전체 실행

### 실제 데이터 연결 후 추가 확인이 필요한 것
- `benefit_type` : 세 source 모두 직접 대응 컬럼 없음 → 파싱 방법 별도 논의 필요
- Youthcenter 연령 컬럼명 : mock에서는 `ageMin`/`ageMax` 사용, 실제 API 확인 필요
- K-Startup `biz_pbanc_nm` : v1.0 노트북의 실제 컬럼 목록과 대조 필요

### 다음 단계: v2
v1.1 데이터가 안정되면 공고문 PDF·HWP 등 첨부파일 본문을 추가합니다.